<img src="logo.png" alt="Vegeta" width="240">

# Fixed-wing drone — airframe, propeller, noise, vibration and life

One notebook for one machine. Part 1 designs the twin-motor airframe (wing load cases, a thicker-wing
revision, whole-aircraft RANS at 4°, a printed nacelle). Part 2 sizes the propeller and the drive on the
drag polar that Part 1 produced (efficiency vs advance ratio, thrust available vs drag, engine-out, a
rotating-frame CFD check with axial inflow, excitations, noise). Part 3 takes the preferred wing through
modal analysis, a Campbell diagram, three mission types, spectra, fatigue and the nacelle vibration
amplitude on resonance. Values pass from part to part as live variables. The OpenFOAM cells run when you
run them (`VEGETA_SKIP_OPENFOAM=1` skips them; Part 2 then falls back to a recorded polar point).

```
Part 1  airframe:   mass budget → parametric CAD → wing load cases (FEA) → two revisions → whole-aircraft CFD → print
Part 2  propeller:  drag polar → BEMT → motor + battery → speed range, climb, engine-out → excitations → noise → CFD check → JSON
Part 3  life:       modes + Campbell → unit stress fields → missions → spectra → damage → resonance amplitude → design comparison
```

# Part 1 — the airframe

In [ ]:
import json, math, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos, aeromant, mellonia, core, boreas, chronos
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.mellonia import viz as mviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM
from vegeta.dedalus.examples import Propeller as PropellerCAD

ROOT = Path("_runs/fixed_wing"); shutil.rmtree(ROOT, ignore_errors=True)
RUNS = ROOT / "airframe"; RUNS.mkdir(parents=True)
RHO, G = 1.2, 9.81                      # air density [kg/m^3], gravity [m/s^2]
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"

## 1. Mission and mass budget

Mapping/patrol drone: 1 m span, cruise around 14 m/s, 3S 5000 mAh battery, 150 g payload. Twin motors
give redundancy (engine-out is a structural case below) and keep the nose free for the payload.
The airframe mass is estimated from the CAD **surface area** — the wing, fuselage and tail are printed
as thin shells — with an explicit areal density; the nacelles are solid-ish and use volume.

In [ ]:
parts = pd.DataFrame([
    ("motors 2212 (2x)",          2 * 55.0),
    ("propellers 9x6 (2x)",       2 * 12.0),
    ("ESC 30 A (2x)",             2 * 25.0),
    ("battery 3S 5000 mAh",       380.0),
    ("flight controller + GPS",   35.0),
    ("servos (4x)",               4 * 12.0),
    ("receiver, wiring, bolts",   45.0),
    ("payload (camera)",          150.0),
], columns=["part", "mass_g"]).set_index("part")

SHELL_AREAL_DENSITY_G_MM2 = 1.1e-4     # 1.1 kg/m^2: ~1.4 mm LW-PLA shell at 0.8 g/cm^3 (assumption)
NACELLE_DENSITY_G_MM3 = 0.5e-3         # PLA at ~40 % effective density (walls + infill), assumption
THRUST_PER_MOTOR_N = 8.0               # static thrust of a 2212 / 9x6 on 3S (datasheet-style value)
BATTERY_WH, USABLE_FRACTION, PROP_EFFICIENCY = 11.1 * 5.0, 0.8, 0.55
parts

## 2. Parametric aircraft (Dedalus)

`FixedWing` builds a NACA `camber/camber_pos/thickness` wing as a constant-chord centre section (inside
the fuselage, with its own faces so it can be clamped in the FEA) plus two tapered outer panels with
dihedral, a revolved fuselage, flat-plate tail surfaces and the two nacelles — built once and mirrored,
so the aircraft is symmetric by construction. `part` chooses `aircraft`, `wing` (wing + nacelles, for the
structure) or `nacelle` (for printing). `angle_of_attack_deg` rotates the whole aircraft for the CFD.

In [ ]:
design_file = RUNS / "fixed_wing.py"
shutil.copy(Path("designs/fixed_wing.py"), design_file)     # the original stays untouched; this copy may be edited
drone = dedalus.load_design(f"{design_file}:FixedWing")
pd.DataFrame(drone.params.table()).set_index("name")

In [ ]:
aircraft = drone.generate()
aircraft          # interactive CadQuery view

In [ ]:
dviz.show(dviz.plot3d(aircraft))

In [ ]:
p0 = drone.resolve()
fig = dviz.plot_sections(aircraft, normal="y", positions=[0.0, p0["nacelle_y"], 0.45 * p0["span"]], cols=3)   # fuselage, nacelle, near tip
fig = dviz.plot_sections(aircraft, normal="x", positions=[-60.0, 60.0, 400.0], cols=3)                          # nose, wing, tail

In [ ]:
b, c0, lam = p0["span"] / 1000, p0["root_chord"] / 1000, p0["taper"]
S = b * c0 * (1 + lam) / 2                       # planform area [m^2] (centre section ~ root chord)
AR = b**2 / S
wing = drone.generate(part="wing"); nacelle = drone.generate(part="nacelle")
airframe_g = (aircraft.surface_area - 2 * nacelle.surface_area) * SHELL_AREAL_DENSITY_G_MM2 + 2 * nacelle.volume * NACELLE_DENSITY_G_MM3
parts.loc["airframe (shells, from CAD area)"] = round(airframe_g, 1)
AUW_kg = parts["mass_g"].sum() / 1000
W = AUW_kg * G
V_CRUISE = 14.0
q = 0.5 * RHO * V_CRUISE**2
print(f"wing area {S:.3f} m^2, aspect ratio {AR:.1f}, airframe {airframe_g:.0f} g, AUW {AUW_kg:.2f} kg")
print(f"wing loading {W / S:.0f} N/m^2, C_L needed at {V_CRUISE} m/s: {W / (q * S):.2f}, "
      f"static thrust-to-weight {2 * THRUST_PER_MOTOR_N / W:.2f}")
parts

## 3. Wing structure (Talos)

The wing + nacelles are analysed as one solid with a **solid-equivalent** modulus for the printed LW-PLA
structure (E = 400 MPa, yield 12 MPa — assumptions to be replaced by coupon tests). Two load cases:

| case | loads | represents |
|---|---|---|
| `pull_up` | lift = 2.5 × W as a uniform pressure on the outer-panel lower skins, + 6 N thrust on each nacelle nose | a 2.5 g pull-up at cruise thrust |
| `engine_out` | lift = 1 × W, 8 N thrust on the **left** nacelle only | full throttle after losing the right motor: asymmetric torsion |

The centre section (|y| < fuselage radius + 5 mm) is clamped — it is glued into the fuselage. The lower
skins are chosen from `talos.inspect_step`: per side, the large B-spline face with the lower centroid.
That choice is printed so you can check it.

In [ ]:
LW_PLA = talos.Material("LW-PLA printed wing (solid-equivalent)", youngs_modulus=400.0, poissons_ratio=0.35,
                        density=0.6e-9, yield_strength=12.0, source="assumed; replace with coupon tests")

def lower_skins(step):
    info = talos.inspect_step(step, units="mm-N-MPa")
    skins = [s for s in info.surfaces if s.kind == "BSpline surface" and s.area > 5e4]
    return [min((s for s in skins if (s.centroid[1] > 0) == right), key=lambda s: s.centroid[2]).tag
            for right in (False, True)]

def wing_regions(rev):
    p = rev.params
    yc, ny, d, f = p["fuselage_diameter"] / 2 + 5.0, p["nacelle_y"], p["nacelle_diameter"] / 2 + 1, p["nacelle_forward"]
    return [talos.SurfacesInBox("root", (-1.0, -yc - 0.1, -100.0, 400.0, yc + 0.1, 100.0)),
            talos.Surfaces("lift", lower_skins(rev.step)),
            talos.SurfacesInBox("motor_left", (-f - 0.1, -ny - d, -d, -f + 0.1, -ny + d, d)),
            talos.SurfacesInBox("motor_right", (-f - 0.1, ny - d, -d, -f + 0.1, ny + d, d))]

def wing_model(rev, loads, name):
    return talos.StructuralModel(rev.step, "mm-N-MPa", LW_PLA, wing_regions(rev), [talos.FixedSupport("root")], loads,
                                 talos.MeshSettings(element_size=10.0), name=name)

def lift_pressure_MPa(n):                       # n x weight spread over the planform, in MPa (N/mm^2)
    return n * W / (S * 1e6)

def pull_up(rev):
    return wing_model(rev, [talos.Pressure("lift", lift_pressure_MPa(2.5)),
                            talos.Force("motor_left", fx=6.0), talos.Force("motor_right", fx=6.0)], "pull_up")

def engine_out(rev):
    return wing_model(rev, [talos.Pressure("lift", lift_pressure_MPa(1.0)), talos.Force("motor_left", fx=8.0)], "engine_out")

print(f"lift pressure at 2.5 g: {lift_pressure_MPa(2.5) * 1e6:.0f} Pa")

In [ ]:
ws = core.Workspace.create(RUNS / "workspace", name="fixed-wing drone")
design = ws.add_design("drone", f"{design_file}:FixedWing")
w1 = design.new_revision(part="wing", note="baseline wing NACA 2412, taper 0.7")
w1.generate()
print("lower skins:", lower_skins(w1.step))
talos.inspect_step(w1.step, units="mm-N-MPa")

In [ ]:
ev_pu = w1.run_fea("pull_up", pull_up, progress=True)
ev_eo = w1.run_fea("engine_out", engine_out, progress=True)
ws.status()

In [ ]:
tviz.show(tviz.plot_problem(pull_up(w1), ev_pu.tool_results[0], arrow_scale=60))

In [ ]:
if ev_pu.ok:
    tviz.show(tviz.plot_results(ev_pu.tool_results[-1], field="von_mises"))

In [ ]:
if ev_pu.ok:
    res = ev_pu.tool_results[-1]
    fig = tviz.plot_section(res, normal="x", origin=(60.0, 0, 0), field="von_mises")      # spanwise: bending stress
    fig = tviz.plot_section(res, normal="y", origin=(0, 250.0, 0), field="von_mises")     # airfoil section at mid-panel
    fig = talos.plot_along_axis(res, axis="y", quantity="displacement", component=2)     # spanwise deflection

In [ ]:
if ev_eo.ok:
    tviz.show(tviz.plot_results(ev_eo.tool_results[-1], field="|U|"))
    print(ev_eo)

### Reading the numbers
`max_displacement` is the tip deflection; `safety_factor_yield` is yield / peak nodal von Mises, which
sits at the clamp or at the nacelle-wing junction (a stress concentration; compare between revisions,
do not read it as an absolute). In `engine_out` look at the *asymmetric* displacement: the thrust twists
the left panel.

## 4. A thicker wing? Two revisions compared

A NACA 2415 (`thickness=0.15`) is stiffer (bending stiffness ~ t³) at a small drag cost. Same load
cases, same factories, the workspace keeps both.

In [ ]:
w2 = next((r for r in ws.revisions() if r.record.get("note") == "NACA 2415: thicker wing"), None) \
     or w1.branch(thickness=0.15, note="NACA 2415: thicker wing")
if not w2.is_generated:
    w2.generate()
for case in tqdm((pull_up, engine_out), desc="load cases"):          # safe to interrupt and re-run
    if w2.evaluation("fea", case.__name__) is None:
        w2.run_fea(case.__name__, case, progress=True)

def row(rev):
    vol_mm3 = rev.geometry_summary()["metrics"]["volume"]
    d = {"rev": rev.id, "note": rev.record.get("note", ""), "thickness": rev.params["thickness"],
         "wing_volume_cm3": vol_mm3 / 1e3}
    for case in ("pull_up", "engine_out"):
        ev = rev.evaluation("fea", case)
        d[f"{case}_tip_mm"] = ev.metrics.get("max_displacement", np.nan) if ev and ev.ok else np.nan
        d[f"{case}_SF"] = ev.metrics.get("safety_factor_yield", np.nan) if ev and ev.ok else np.nan
    return d

table = pd.DataFrame([row(r) for r in (w1, w2)]).set_index("rev").round(2)
table

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
table.plot.bar(y=["pull_up_tip_mm", "engine_out_tip_mm"], ax=ax[0], title="tip deflection [mm]", rot=0)
table.plot.bar(y=["pull_up_SF", "engine_out_SF"], ax=ax[1], title="safety factor (yield)", rot=0)
fig.tight_layout()
best = table["pull_up_SF"].idxmax()
preferred_wing = ws.revision(best)
preferred_wing.label("preferred", note=f"stiffer; pull-up SF {table.loc[best, 'pull_up_SF']}")
ws.status()

## 5. Whole-aircraft aerodynamics at 4° (Aeromant)

Steady RANS (k-ω SST) of the complete aircraft with the preferred wing thickness, 14 m/s, 4° angle of
attack, coarse mesh (a few minutes). The template's domain scales with `reference_length`, which
must be ≥ span/4 here (`max_body_extent`), so **Cm is referenced to 0.25 m**; Cl and Cd use the wing
area. The engineering outputs: lift and drag at this attitude, the level-flight speed at which this
Cl carries the aircraft, and cruise power → endurance.

Skipped when `VEGETA_SKIP_OPENFOAM=1` is set; Part 2 then uses a recorded polar point instead of this run's.

In [ ]:
a1 = preferred_wing.branch(part="aircraft", angle_of_attack_deg=4.0, note="whole aircraft, 4 deg AoA, for CFD")
a1.generate(stl_tolerance=0.2)
cases = {}

def cruise_4deg(rev, workdir):
    case = aeromant.CFDCase(
        "rans_ksst_external", rev.stl,
        dict(velocity=V_CRUISE, kinematic_viscosity=1.5e-5, density=RHO, reference_area=S, reference_length=0.25,
             center_of_rotation=(0.05, 0.0, 0.0), iterations=400, residual_target=1e-4,
             surface_level=4, near_level=3, wake_level=2, cells_per_length=2.0),
        workdir=workdir, geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())
    cases[rev.id] = case
    return case

ev_cfd = a1.run_cfd("cruise_4deg", cruise_4deg, progress=True) if RUN_CFD else None
print(ev_cfd if ev_cfd is not None else "CFD skipped (VEGETA_SKIP_OPENFOAM=1): run this cell on a machine with OpenFOAM")

In [ ]:
if ev_cfd is not None:
    case = cases[a1.id]
    aviz.show(aviz.plot_setup(case))

In [ ]:
if ev_cfd is not None and ev_cfd.ok:
    aviz.show(aviz.plot_mesh_slice(case, normal="y", origin=(0, 0.0, 0)))

In [ ]:
if ev_cfd is not None and ev_cfd.ok:
    aviz.show(aviz.plot_field_slice(case, "U", normal="y", origin=(0, p0["nacelle_y"] / 1000, 0)))   # through a nacelle
    fig = aviz.plot_section(case, "p", normal="y", origin=(0, 0.0, 0), zoom=2)                       # fuselage plane
    fig = aviz.plot_section(case, "U", normal="x", origin=(0.10, 0, 0), zoom=1.5)                    # behind the leading edge: wing + nacelles

In [ ]:
if ev_cfd is not None and ev_cfd.ok:
    aviz.show(aviz.plot_streamlines(case, n=80))

In [ ]:
if ev_cfd is not None and ev_cfd.ok:
    aviz.show(aviz.plot_surface_pressure(case))
    fig = aeromant.plot_coefficients(case.workdir)

In [ ]:
if ev_cfd is not None and ev_cfd.ok:
    m = ev_cfd.metrics
    Cl, Cd = m["Cl"], m["Cd"]
    V_level = math.sqrt(2 * W / (RHO * S * Cl)) if Cl > 0 else float("nan")   # speed at which this Cl carries W
    D_level = 0.5 * RHO * V_level**2 * S * Cd
    P_cruise = D_level * V_level / PROP_EFFICIENCY                              # electrical-ish power at the props
    endurance_min = BATTERY_WH * USABLE_FRACTION / P_cruise * 60 if P_cruise > 0 else float("nan")
    aero = pd.Series({
        "Cl (4 deg)": Cl, "Cd (4 deg)": Cd, "L/D": Cl / Cd,
        "lift at 14 m/s [N]": m["lift_force_N"], "weight [N]": W,
        "level-flight speed at this Cl [m/s]": V_level, "drag there [N]": D_level,
        "cruise power (prop eff. 0.55) [W]": P_cruise, "endurance (80 % of battery) [min]": endurance_min,
        "converged": m["converged"], "cells": m["mesh_cells"],
    })
    print(aero.to_string())

The mesh is coarse and the run short: treat these as a **first estimate** (Cl at 4° should land in the
0.4–0.6 range for this wing; the trailing edge is not resolved). The angle of attack is a design
parameter, so a sweep is three more branches (`angle_of_attack_deg=0, 8, 12`) and one table — a job for
the comparison tooling of Stage 11, or for a loop you write here.

## 6. Print a nacelle (Mellonia)

Nose down on the bed: the chamfered motor face is flat, so `rotate_y=90` puts it on the plate. Two
copies are needed — slice once, print twice.

In [ ]:
n1 = design.new_revision(part="nacelle", note="nacelle for printing")
n1.generate()
ev_p = n1.run_print("nose_down", GENERIC_PLA_0_2MM, mellonia.Orientation(rotate_y=90))
print(ev_p)

In [ ]:
if ev_p.ok:
    prn = ev_p.tool_results[0]
    mviz.show(mviz.plot_toolpath(prn))
    fig = mviz.plot_layer_grid(prn, n=6, cols=3)

## 7. Where we are

One design file, four revisions, six evaluations — every number above can be traced to a folder with
the tool's native files and the exact commands that produced it.

In [ ]:
ws.status()

In [ ]:
for p in sorted((RUNS / "workspace" / "revisions" / a1.id).rglob("*"))[:20]:
    print(p.relative_to(RUNS / "workspace"))

**Next steps an engineer would take:** an angle-of-attack sweep and the tail sizing from Cm; a carbon
spar (a second solid in the wing, bonded); a proper printed-shell wing model (Talos is a solid-element
tool — a shell analysis is a different model); replacing the assumed LW-PLA properties with coupon
tests; and letting the AI copilot (notebook 07) propose weight savings on `_runs/fixed_wing/fixed_wing.py`,
each proposal built and measured before you accept it.

# Part 2 — the propeller and the drive

In [ ]:
RUNS = ROOT / "propeller"; RUNS.mkdir()
AUW_KG = AUW_kg                                        # Part 1 mass budget
WING_AREA = S                                          # m^2, Part 1 planform
MOTORS = 2
V_CFD = V_CRUISE
if ev_cfd is not None and ev_cfd.ok:
    CL_CFD, CD_CFD = ev_cfd.metrics["Cl"], ev_cfd.metrics["Cd"]
    print(f"polar point from this run's whole-aircraft CFD at 4 deg: Cl {CL_CFD:.3f}, Cd {CD_CFD:.4f}")
else:
    CL_CFD, CD_CFD = 0.40, 0.070                       # recorded from an earlier run of the Part 1 CFD (coarse mesh)
    print(f"CFD not run: recorded polar point Cl {CL_CFD}, Cd {CD_CFD} at 4 deg (rerun Part 1's CFD to refresh it)")

## 1. Hardware and the aircraft drag polar

The CFD gave one point (Cl, Cd at 4°). A parabolic polar `Cd = Cd0 + k Cl²` is fitted through it with an
assumed Oswald factor, so drag can be evaluated at any speed. That is an assumption to replace with a
CFD angle-of-attack sweep (three more revisions in notebook 09).

In [ ]:
d, p = boreas.inches(9, 6)
prop = boreas.Propeller.from_pitch("9x6 electric", d, p, blades=2, chord_root_m=0.014, chord_max_m=0.022, chord_tip_m=0.006,
                                   mass_kg=0.012, rotor_mass_kg=0.045, notes="generic planform")
airfoil = boreas.Airfoil(name="thin cambered section", cl_alpha=2 * math.pi * 0.9, alpha0_deg=-2.5, cl_max=1.1, cd0=0.02, k=0.04,
                         source="assumed for a 9-inch blade at Re ~ 1.5e5")
motor = boreas.Motor("2212-920KV", kv_rpm_per_volt=920, resistance_ohm=0.12, no_load_current_a=0.6, max_current_a=20, mass_kg=0.055)
battery = boreas.Battery("3S 5000 mAh", cells=3, capacity_ah=5.0, usable_fraction=0.8, mass_kg=0.380)
system = boreas.Propulsion(prop, airfoil, motor, battery, rho=RHO)

AR, OSWALD = 5.9, 0.8
K_INDUCED = 1 / (math.pi * AR * OSWALD)
CD0 = CD_CFD - K_INDUCED * CL_CFD**2
W = AUW_KG * 9.81

def drag(v):                       # level flight: lift = weight -> Cl(v) -> Cd(v) -> D
    q = 0.5 * RHO * v**2
    cl = W / (q * WING_AREA)
    return q * WING_AREA * (CD0 + K_INDUCED * cl**2), cl

print(f"polar: Cd = {CD0:.4f} + {K_INDUCED:.4f} Cl^2  (through the CFD point Cl {CL_CFD}, Cd {CD_CFD})")
print(f"drag at {V_CFD} m/s level flight: {drag(V_CFD)[0]:.2f} N total, {drag(V_CFD)[0] / MOTORS:.2f} N per motor")

## 2. The propeller, drawn

In [ ]:
r = np.array(prop.r); c = np.array(prop.chord); beta = np.array(prop.beta_deg)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].fill_between(r * 1000, -c * 1000 * 0.3, c * 1000 * 0.7, color="#9fb8d0"); ax[0].set_aspect("equal")
ax[0].set(xlabel="radius [mm]", ylabel="chord [mm]", title=f"planform, solidity {prop.solidity:.3f}"); ax[0].grid(alpha=0.3)
ax[1].plot(r * 1000, beta, "o-"); ax[1].set(xlabel="radius [mm]", ylabel="blade angle β [deg]", title=f"twist for {p * 1000:.0f} mm pitch"); ax[1].grid(alpha=0.3)
fig.tight_layout()
CAD_KW = dict(diameter=d * 1000, pitch=p * 1000, blades=2, hub_diameter=16, hub_height=9, bore=5, chord_root=14, chord_max=22, chord_tip=6, thickness=0.09, camber=0.04)
cad = PropellerCAD().generate(**CAD_KW)
prop_files = cad.export(RUNS / "fw_9x6_cad", formats=("step", "stl"), stl_tolerance=0.02)
dviz.show(dviz.plot3d(cad))

In [ ]:
fig = dviz.plot_sections(cad, normal="x", positions=[30.0, 65.0, 105.0], cols=3)

## 3. Efficiency against advance ratio — the fixed-wing question

`J = V / (n D)`. The propeller is efficient in a band of J; the cruise point should sit near the peak.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for rpm in (5000, 7000, 9000):
    Vs = np.linspace(0.5, 22, 30)
    ops = [boreas.solve(prop, airfoil, rpm, v, RHO) for v in Vs]
    J = [o.advance_ratio for o in ops]
    ax[0].plot(J, [o.efficiency for o in ops], label=f"{rpm} rpm")
    ax[1].plot(J, [o.ct for o in ops], label=f"{rpm} rpm")
ax[0].set(xlabel="advance ratio J", ylabel="propeller efficiency", ylim=(0, 1), title="η(J)"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].set(xlabel="advance ratio J", ylabel="Ct", title="thrust coefficient"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()

## 4. Thrust available vs drag: speed range, climb margin, engine-out

Thrust available at full throttle (both motors) against the drag curve gives the maximum level speed;
the gap at cruise is the climb margin (`rate of climb ≈ (T − D) V / W`). With one motor out, the
remaining motor's full-throttle thrust must still exceed drag somewhere, or the aircraft cannot hold
altitude.

In [ ]:
Vs = np.linspace(8, 26, 19)
D = np.array([drag(v)[0] for v in Vs])
T_full = np.array([system.at_throttle(1.0, v).thrust for v in Vs])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(Vs, D, color="#c62828", label="drag (level flight)")
ax[0].plot(Vs, MOTORS * T_full, label="thrust, 2 motors full throttle")
ax[0].plot(Vs, T_full, "--", label="thrust, 1 motor (engine out)")
ax[0].set(xlabel="airspeed [m/s]", ylabel="force [N]", title="thrust available vs required"); ax[0].legend(); ax[0].grid(alpha=0.3)
roc = (MOTORS * T_full - D) * Vs / W
ax[1].plot(Vs, roc); ax[1].axhline(0, color="k", lw=0.8)
ax[1].set(xlabel="airspeed [m/s]", ylabel="rate of climb [m/s]", title="climb at full throttle"); ax[1].grid(alpha=0.3)
fig.tight_layout()
stall_v = math.sqrt(2 * W / (RHO * WING_AREA * 1.1))
v_max = Vs[np.where(MOTORS * T_full > D)[0].max()] if np.any(MOTORS * T_full > D) else float("nan")
eo = np.where(T_full > D)[0]
print(f"stall speed (Cl_max 1.1 assumed): {stall_v:.1f} m/s | max level speed ~{v_max:.0f} m/s | best climb {roc.max():.1f} m/s at {Vs[roc.argmax()]:.0f} m/s")
print("engine out: " + (f"level flight possible between {Vs[eo.min()]:.0f} and {Vs[eo.max()]:.0f} m/s" if len(eo) else "NOT possible on one motor"))

## 5. Cruise, climb and full-power points; endurance and range

In [ ]:
D_cruise = drag(V_CFD)[0]
cruise = system.for_thrust(D_cruise / MOTORS, V_CFD)
climb = system.at_throttle(1.0, 12.0)
static = system.at_throttle(1.0, 0.0)
one_engine = system.at_throttle(1.0, V_CFD)
P_cruise = MOTORS * cruise.electrical_power
endurance_min = battery.usable_wh / P_cruise * 60
range_km = V_CFD * endurance_min * 60 / 1000
pts = {"cruise 14 m/s": cruise, "climb 12 m/s full": climb, "static full": static, "engine-out cruise (1 motor full)": one_engine}
tbl = pd.DataFrame({k: {"throttle": v.throttle, "rpm": v.rpm, "thrust_N": v.thrust, "current_A": v.current, "electrical_W": v.electrical_power,
                        "prop_eff": v.aero.efficiency, "motor_eff": v.motor_efficiency, "J": v.aero.advance_ratio, "current_limited": v.current_limited}
                    for k, v in pts.items()}).round(3)
print(f"cruise: {P_cruise:.0f} W electrical for both motors -> {endurance_min:.0f} min, {range_km:.0f} km still-air range on {battery.usable_wh:.0f} Wh")
print(f"overall propulsive efficiency at cruise: {cruise.aero.efficiency * cruise.motor_efficiency:.2f} (prop x motor)")
tbl

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(cruise.aero.r * 1000, cruise.aero.dT_dr, label="cruise"); ax[0].plot(climb.aero.r * 1000, climb.aero.dT_dr, label="climb")
ax[0].set(xlabel="radius [mm]", ylabel="dT/dr [N/m]", title="thrust loading"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(cruise.aero.r * 1000, cruise.aero.alpha_deg, label="cruise"); ax[1].plot(climb.aero.r * 1000, climb.aero.alpha_deg, label="climb")
ax[1].set(xlabel="radius [mm]", ylabel="angle of attack [deg]", title="section incidence"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()

## 6. What the wing and nacelle will feel

1P and 2P (blade-pass) lines against rpm, and the unbalance force on the nacelle. The engine-out case
matters twice: the remaining motor runs at full rpm (highest excitation), and the thrust is asymmetric
(notebook 09's `engine_out` load case).

In [ ]:
rr = np.linspace(3000, 10000, 15)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(rr, rr / 60, label="1P"); ax[0].plot(rr, 2 * rr / 60, label="2P (blade pass)")
for name, pt in (("cruise", cruise), ("climb", climb)):
    ax[0].axvline(pt.rpm, color="#888", ls=":"); ax[0].text(pt.rpm, 20, name, rotation=90, va="bottom")
ax[0].set(xlabel="rpm", ylabel="frequency [Hz]"); ax[0].legend(); ax[0].grid(alpha=0.3)
for g in (6.3, 2.5):
    ax[1].plot(rr, [boreas.unbalance_force(prop.rotor_mass_kg, x, g) for x in rr], label=f"G {g}")
ax[1].set(xlabel="rpm", ylabel="rotating force [N]", title=f"unbalance, rotor {prop.rotor_mass_kg * 1000:.0f} g"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()
pd.DataFrame({k: boreas.excitations(prop, v.rpm) for k, v in pts.items()}).round(2)

### Noise: blade-passing tones and broadband

Gutin's steady-loading tones at 1 m broadside (dB re 20 µPa) plus a broadband allowance, per operating
point; two motors add 3 dB, distance takes 20 log10(d) off. A two-blade propeller at ~6000 rpm puts the
blade-passing tone near 200 Hz — the tonal component is what an observer notices at a distance.

In [ ]:
DIST, ANGLE = 1.0, 90.0
noise_rows = {}
for name, pt in pts.items():
    tones = boreas.gutin_harmonics(prop, pt.thrust, pt.aero.torque, pt.rpm, DIST, ANGLE, boreas.AIR, harmonics=6)
    bb = boreas.broadband_level(prop, pt.thrust, pt.rpm, DIST, boreas.AIR)
    one = 10 * math.log10(10 ** (tones["total_tonal_db"] / 10) + 10 ** (bb / 10))
    n_running = 1 if "engine-out" in name else MOTORS
    noise_rows[name] = {"rpm": pt.rpm, "BPF_hz": tones["blade_pass_hz"], "tonal_dB": tones["total_tonal_db"], "broadband_dB": bb,
                        "one_prop_dB_at_1m": one, "aircraft_dB_at_1m": one + 10 * math.log10(n_running),
                        "aircraft_dB_at_100m": one + 10 * math.log10(n_running) - 40}
    if name.startswith("cruise"):
        fig, ax = plt.subplots(figsize=(6.5, 3.4))
        ax.bar(tones["frequency_hz"], tones["spl_db"], width=12, label="Gutin tones, cruise, one propeller")
        ax.axhline(bb, color="#c62828", ls="--", label="broadband allowance"); ax.set(xlabel="frequency [Hz]", ylabel="dB re 20 µPa at 1 m"); ax.legend(); ax.grid(alpha=0.3)
noise = pd.DataFrame(noise_rows).T.round(1)
noise

## 7. Export — the hand-off

In [ ]:
grid = boreas.performance_map(prop, airfoil, np.linspace(3000, 10000, 8), np.linspace(0, 24, 7), RHO)
res = boreas.export(RUNS / "fw_9x6.json", prop, airfoil, map=grid, motor=motor, battery=battery,
                    points={"cruise": cruise, "climb": climb, "static": static, "engine_out": one_engine},
                    notes=f"fixed wing from notebook 09; AUW {AUW_KG} kg, {MOTORS} motors; polar Cd0 {CD0:.4f}, k {K_INDUCED:.4f}; "
                          f"cruise endurance {endurance_min:.0f} min, range {range_km:.0f} km")
print(res)
doc = boreas.load(res.artifacts["json"])
print("files:", sorted(f.name for f in RUNS.glob("fw_9x6*")))
pd.DataFrame({k: {"rpm": v["rpm"], "thrust_N": v["aero"]["thrust"], "current_A": v["current"], "prop_eff": v["aero"]["efficiency"],
                  "shaft_hz": v["excitation"]["shaft_hz"], "blade_pass_hz": v["excitation"]["blade_pass_hz"],
                  "unbalance_N": v["excitation"]["unbalance_force_n"]} for k, v in doc["points"].items()}).round(2)

## 8. CFD check of the cruise point — OpenFOAM, rotating reference frame with axial inflow

`aeromant`'s `rotor_mrf` template: the same CAD propeller in a rotating cell zone with the cruise
airspeed as inflow (steady k-ω SST, MRF). The check is coarse (about 100 k cells, minutes on one core);
expect tens of percent against blade element theory and read the sign message. The CAD axis (Z) is
rotated to +x first. `VEGETA_SKIP_OPENFOAM=1` skips the run when executing headlessly.

In [ ]:
prop_x = dedalus.Geometry.from_cadquery(cad.shape.rotate((0, 0, 0), (0, 1, 0), 90), name="prop_axis_x")
stl_x = prop_x.export_stl(RUNS / "fw_9x6_cad" / "prop_axis_x.stl", tolerance=0.02)
CFD_PARAMS = dict(rpm=cruise.rpm, airspeed=V_CFD, diameter=d, kinematic_viscosity=1.5e-5, density=RHO, rotation=1, iterations=400,
                  cells_per_diameter=6.0, surface_level=3, near_level=2, rotor_level=2, wake_level=1)
cfd = None
if RUN_CFD:
    case = aeromant.CFDCase("rotor_mrf", stl_x, CFD_PARAMS, workdir=RUNS / "fw_9x6_cfd_cruise", geometry_units="mm",
                            environment=aeromant.OpenFOAMEnvironment.detect())
    print(case.prepare(overwrite=True))
    cfd = case.run(progress=True)
    print(cfd)
else:
    print("CFD skipped (VEGETA_SKIP_OPENFOAM=1): run this cell on a machine with OpenFOAM to get the check")

In [ ]:
if cfd is not None and cfd.ok:
    m = cfd.metrics
    compare = pd.DataFrame({"BEMT (Boreas)": {"thrust_N": cruise.thrust, "torque_Nm": cruise.aero.torque, "power_W": cruise.aero.power, "efficiency": cruise.aero.efficiency},
                            "CFD (rotor_mrf)": {"thrust_N": m["thrust_N"], "torque_Nm": m["torque_Nm"], "power_W": m["power_W"], "efficiency": m["efficiency"]}})
    compare["CFD / BEMT"] = compare["CFD (rotor_mrf)"] / compare["BEMT (Boreas)"]
    print(f"{m['mesh_cells']} cells, {'converged' if m['converged'] else 'not converged'} in {m['iterations']} iterations; J = {m['advance_ratio']:.2f}")
    display(compare.round(3))

In [ ]:
if cfd is not None and cfd.ok:
    aviz.show(aviz.plot_field_slice(case, "U", normal="z"))
    fig = aviz.plot_section(case, "p", normal="z", zoom=2)
    aviz.show(aviz.plot_streamlines(case, n=80, normal_plane="z"))
    aviz.show(aviz.plot_surface_pressure(case))

### The flow as a video

Tracer particles carried through the converged velocity field (a steady result played as motion), the rotor turned at its rpm for the eye; written with OpenCV. Open the file with any player if the inline video does not show.

In [ ]:
if cfd is not None and cfd.ok:
    from IPython.display import Video
    video = aviz.animate_particles(case, RUNS / "fw_9x6_cruise_flow.mp4", rpm=cruise.rpm, seconds=6, fps=24)
    display(Video(str(video), embed=False, width=720))

### A particle movie from the OpenFOAM result

A few tracer particles carried through the converged velocity field, in a side view and along the axis, the blades turning in consistent slow motion (every frame the rotor turns 10° and the flow advances by the same time). `vegeta.aeromant.movie` reads the case once and draws with OpenCV; the swirl behind the disc is the swirl the rotor puts in.

In [ ]:
if cfd is not None and cfd.ok:
    from IPython.display import Video
    from vegeta.aeromant import movie
    clip = movie.make_movie(case, RUNS / "fw_9x6_cruise_particles.mp4", blades=prop.blades, n=40, seconds=12, fps=24, degrees_per_frame=10,
                            title="9x6 propeller at cruise (rotor_mrf, air): tracer particles")
    display(Video(str(clip), embed=False, width=900))

## 9. The blade under load (Talos), and the animation

One blade with its hub, hub faces fixed, the blade's share of the full-throttle thrust and of the torque
(at 0.7 R) applied as tractions over the blade — the right totals with an approximate distribution.
A moulded glass-filled nylon blade (E = 8 GPa, yield 100 MPa) — an assumption to replace with the real material. The video ramps the load from zero to full while the blade turns at the full-throttle
rpm: a linear static result presented in motion, not a transient analysis.

In [ ]:
blade = PropellerCAD().generate(**dict(CAD_KW, blades=1))
bfiles = blade.export(RUNS / "fw_9x6_blade", formats=("step",))
hub_r, hub_h, R_tip = CAD_KW["hub_diameter"] / 2, CAD_KW["hub_height"], CAD_KW["diameter"] / 2
BLADE_REGIONS = [talos.SurfacesInBox("hub", (-hub_r - 0.5, -hub_r - 0.5, -hub_h / 2 - 0.5, hub_r + 0.5, hub_r + 0.5, hub_h / 2 + 0.5)),
                 talos.SurfacesInBox("blade", (hub_r - 1.5, -R_tip, -R_tip, R_tip + 1.0, R_tip, R_tip))]
BLADE_MAT = talos.Material("PA6-GF30 (moulded blade)", youngs_modulus=8000.0, poissons_ratio=0.35, density=1.35e-9, yield_strength=100.0, source="assumed moulded glass-filled nylon")
T_blade = climb.thrust / prop.blades                                     # N per blade at full throttle
F_tan = climb.aero.torque / (prop.blades * 0.7 * prop.radius)           # tangential force per blade at 0.7 R
blade_model = talos.StructuralModel(bfiles.artifacts["step"], "mm-N-MPa", BLADE_MAT, BLADE_REGIONS, [talos.FixedSupport("hub")],
                                    [talos.Force("blade", fz=T_blade, fy=-F_tan)], talos.MeshSettings(element_size=1.2), name="blade_full_throttle")
blade_mesh = blade_model.mesh(RUNS / "fw_9x6_blade_fea", progress=True)
res_blade = blade_model.solve(RUNS / "fw_9x6_blade_fea", progress=True)
print(f"per blade at full throttle ({climb.rpm:.0f} rpm): thrust {T_blade:.2f} N, tangential {F_tan:.2f} N")
print(res_blade)
if res_blade.ok:
    tviz.show(tviz.plot_results(res_blade, field="von_mises"))
    from IPython.display import Video
    video = tviz.animate(res_blade, RUNS / "fw_9x6_blade_stress.mp4", rpm=climb.rpm, axis="z", seconds=5, fps=24)
    display(Video(str(video), embed=False, width=720))

In [ ]:
if cfd is not None and cfd.ok:
    doc = json.loads((RUNS / "fw_9x6.json").read_text())
    doc["cfd_cruise"] = {"template": "rotor_mrf", "parameters": CFD_PARAMS, "metrics": {k: v for k, v in cfd.metrics.items() if not isinstance(v, (list, dict))}}
    (RUNS / "fw_9x6.json").write_text(json.dumps(doc, indent=2, default=float))
    print("cfd_cruise added to", RUNS / "fw_9x6.json")

**Hand-off:** Part 3 takes the cruise, climb and engine-out points straight from these objects; the JSON is the same record for anything outside this notebook.

# Part 3 — vibration, cyclic loads and life

In [ ]:
RUNS = ROOT / "life"; RUNS.mkdir()
PREFERRED = dict(thickness=preferred_wing.params["thickness"])       # Part 1: the preferred wing revision
AUW_KG, WING_AREA_M2 = AUW_kg, S
PROP = {name: {"rpm": pt.rpm, "thrust_N": pt.thrust, "unbalance_N": boreas.excitations(prop, pt.rpm)["unbalance_force_n"]}
        for name, pt in (("cruise", cruise), ("climb", climb), ("engine_out", one_engine))}       # Part 2 operating points
BLADES = prop.blades
NACELLE_MASS_T = (motor.mass_kg + prop.mass_kg + 0.025) * 1e-3     # motor + prop + ESC per nacelle, tonnes
W = AUW_KG * 9.81
LIFT_UNIT_MPA = W / (WING_AREA_M2 * 1e6)                      # pressure for n = 1 (level flight) in MPa
print(f"preferred wing {PREFERRED} | nacelle mass {NACELLE_MASS_T * 1e6:.0f} g")
pd.DataFrame(PROP).round(2)

## 1. Wing, masses, one mesh

In [ ]:
p = drone.resolve(part="wing", **PREFERRED)
wing = drone.generate(part="wing", **PREFERRED)
cad = wing.export(RUNS / "cad")             # LW_PLA: the material of Part 1

def lower_skins(step):
    info = talos.inspect_step(step, units="mm-N-MPa")
    skins = [s for s in info.surfaces if s.kind == "BSpline surface" and s.area > 5e4]
    return [min((s for s in skins if (s.centroid[1] > 0) == right), key=lambda s: s.centroid[2]).tag for right in (False, True)]

yc, ny, d, f = p["fuselage_diameter"] / 2 + 5.0, p["nacelle_y"], p["nacelle_diameter"] / 2 + 1, p["nacelle_forward"]
REGIONS = [talos.SurfacesInBox("root", (-1.0, -yc - 0.1, -100.0, 400.0, yc + 0.1, 100.0)),
           talos.Surfaces("lift", lower_skins(cad.artifacts["step"])),
           talos.SurfacesInBox("motor_left", (-f - 0.1, -ny - d, -d, -f + 0.1, -ny + d, d)),
           talos.SurfacesInBox("motor_right", (-f - 0.1, ny - d, -d, -f + 0.1, ny + d, d))]
MASSES = [talos.PointMass("motor_left", NACELLE_MASS_T), talos.PointMass("motor_right", NACELLE_MASS_T)]
MESH = talos.MeshSettings(element_size=12.0)

def model(loads, name, step=None):
    return talos.StructuralModel(step or cad.artifacts["step"], "mm-N-MPa", LW_PLA, REGIONS, [talos.FixedSupport("root")],
                                 loads, MESH, name=name, masses=MASSES)

base = model([], "modal")
print(base.mesh(RUNS / "mesh", progress=True))

def case_dir(name):
    d_ = RUNS / name
    if not d_.exists():
        shutil.copytree(RUNS / "mesh", d_)
    return d_

## 2. Modes with the nacelles on the wing, and the Campbell diagram

The frequency diagram: 1P and blade-pass lines against rpm, the wing modes as horizontal lines, the operating points marked.

In [ ]:
modes = base.solve_modes(case_dir("modal"), n_modes=8, progress=True)
freqs = modes.metrics["frequencies_hz"]
print([round(x, 1) for x in freqs])
tviz.show(tviz.plot_mode(modes, mode=1))     # first wing bending

In [ ]:
tviz.show(tviz.plot_mode(modes, mode=7))     # the mode near the cruise shaft frequency

In [ ]:
structure = chronos.Structure(tuple(freqs), damping_ratio=0.03, source="Talos modal, LW-PLA solid-equivalent E, nacelle masses")
fig = structure.campbell({"1P": 1, "2P": BLADES}, np.linspace(2000, 10000, 30), operating_rpm={k: v["rpm"] for k, v in PROP.items()})
lines = {k: v["rpm"] / 60 for k, v in PROP.items()}
margins = pd.DataFrame({k: {"1P_hz": fq, "nearest_mode_hz": structure.nearest_mode(fq), "margin": structure.margin(fq),
                            "amplification": float(structure.amplification(fq)[0])} for k, fq in lines.items()}).round(2)
margins

### A finding, and a decision

The cruise shaft frequency (≈ 99 Hz) falls within a few percent of a wing mode: the dynamic
amplification is an order of magnitude, the usual comfort margin is 20 %. Three ways out, all
engineer decisions, none of them a change the software makes on its own:

1. **move the excitation** — cruise at a different rpm (a different propeller pitch, notebook 12);
2. **move the mode** — a stiffer or lighter nacelle mount, a spar;
3. **accept and verify** — run the fatigue with the amplification and see whether the life is still acceptable.

This notebook does (3) with the amplified vibration in the spectrum, and at the end repeats the
assessment for the thinner NACA 2412 wing to show how far a design change moves the numbers.

## 3. Unit load cases

| pattern | unit case | level unit |
|---|---|---|
| `lift` | pressure for n = 1 on the outer-panel lower skins | load factor n |
| `thrust` | +6 N on both nacelle noses | N per motor |
| `thrust_left` | +8 N on the left nacelle only (engine-out) | N |
| `vib_left` | 1 N vertical at the left nacelle nose (rotor unbalance) | N |

In [ ]:
UNIT = {
    "lift":        (model([talos.Pressure("lift", LIFT_UNIT_MPA)], "lift"), 1.0),
    "thrust":      (model([talos.Force("motor_left", fx=6.0), talos.Force("motor_right", fx=6.0)], "thrust"), 6.0),
    "thrust_left": (model([talos.Force("motor_left", fx=8.0)], "thrust_left"), 8.0),
    "vib_left":    (model([talos.Force("motor_left", fz=1.0)], "vib_left"), 1.0),
}
unit_results = {}
for name, (m, load) in tqdm(UNIT.items(), desc="unit cases"):
    unit_results[name] = m.solve(case_dir(name))
    r = unit_results[name]
    print(f"{name:<12} {'ok' if r.ok else 'FAILED'}  max von Mises {r.metrics.get('max_von_mises', float('nan')):.3f} MPa at level {load:g}")
unit_cases = {k: (unit_results[k], UNIT[k][1]) for k in UNIT}
tviz.show(tviz.plot_results(unit_results["vib_left"], field="von_mises"))

## 4. Three missions

`lift` levels are load factors (1 = level flight, 1.4 = a 45° banked turn, 2.5 = the pull-up of
notebook 09); gusts are `repeat`ed excursions. Vibration: unbalance on the left nacelle at the shaft
frequency of the segment's rpm (right nacelle assumed identical by symmetry — the spectrum counts one
side; the hotspot map shows where).

In [ ]:
def unb(point):
    return chronos.Excitation(f"unbalance {point}", PROP[point]["rpm"] / 60, PROP[point]["unbalance_N"], "vib_left")

TC, TCL = PROP["cruise"]["thrust_N"], PROP["climb"]["thrust_N"]
missions = {
    "survey": chronos.Mission("survey", (
        chronos.Segment("take-off + climb", 60, {"lift": 1.2, "thrust": TCL}, (unb("climb"),)),
        chronos.Segment("mapping legs", 2400, {"lift": 1.0, "thrust": TC}, (unb("cruise"),)),
        chronos.Segment("turn between legs", 8, {"lift": 1.3, "thrust": TC}, (unb("cruise"),), repeat=16),
        chronos.Segment("light turbulence", 2, {"lift": 1.25, "thrust": TC}, (unb("cruise"),), repeat=80),
        chronos.Segment("descent + landing", 90, {"lift": 0.9, "thrust": 0.3}),
        chronos.Segment("touchdown", 1, {"lift": 1.8}),
    ), "45 min mapping survey: long straight legs, gentle turns"),
    "patrol": chronos.Mission("patrol", (
        chronos.Segment("take-off + climb", 60, {"lift": 1.2, "thrust": TCL}, (unb("climb"),)),
        chronos.Segment("loiter", 1500, {"lift": 1.15, "thrust": TC}, (unb("cruise"),)),
        chronos.Segment("steep turn", 6, {"lift": 1.6, "thrust": TC}, (unb("cruise"),), repeat=60),
        chronos.Segment("evasive pull-up", 2, {"lift": 2.5, "thrust": TCL}, (unb("climb"),), repeat=4),
        chronos.Segment("engine-out drill", 30, {"lift": 1.0, "thrust_left": PROP["engine_out"]["thrust_N"]}, (unb("engine_out"),)),
        chronos.Segment("descent + landing", 90, {"lift": 0.9, "thrust": 0.3}),
        chronos.Segment("touchdown", 1, {"lift": 2.0}),
    ), "30 min patrol: continuous loiter turns, a few hard pull-ups, one engine-out drill"),
    "windy_hops": chronos.Mission("windy_hops", (
        chronos.Segment("take-off + climb", 45, {"lift": 1.3, "thrust": TCL}, (unb("climb"),), repeat=6),
        chronos.Segment("short cruise", 300, {"lift": 1.0, "thrust": TC}, (unb("cruise"),), repeat=6),
        chronos.Segment("gust", 1.5, {"lift": 1.7, "thrust": TC}, (unb("cruise"),), repeat=300),
        chronos.Segment("hard touchdown", 1, {"lift": 2.5}, repeat=6),
    ), "six short hops in gusty wind: 300 gusts, six hard touchdowns, ~35 min"),
}
for m in missions.values():
    fig = m.profile(patterns=["lift", "thrust", "thrust_left"])
pd.DataFrame({k: {"duration_min": m.duration_h * 60, "segments": len(m.segments)} for k, m in missions.items()}).T

In [ ]:
spectra = {k: chronos.build_spectrum(m, structure) for k, m in missions.items()}
for k, sp in spectra.items():
    sp.save(RUNS / f"spectrum_{k}.json")
    fig = sp.plot()
spectra["patrol"].table().round(4)

## 5. Damage per mission, hotspot, static re-check

S-N for LW-PLA — **assumed**: σ_f = 22 MPa, b = −0.12, Goodman with 14 MPa ultimate. Printed foamed
PLA is weak in fatigue and sensitive to temperature; coupon tests first.

In [ ]:
CURVE = talos.FatigueCurve("LW-PLA (assumed)", sigma_f=22.0, b=-0.12, ultimate=14.0, source="assumed; coupon tests needed")
fatigue = {k: talos.assess_fatigue(unit_cases, spectra[k].to_dict(), CURVE, workdir=RUNS / f"fatigue_{k}") for k in tqdm(missions, desc="fatigue")}
life = pd.DataFrame({k: {"duration_min": missions[k].duration_h * 60, "damage_per_mission": f.result.metrics["damage_per_pass"],
                         "missions_to_failure": f.result.metrics["passes_to_failure"], "hours_to_failure": f.result.metrics["hours_to_failure"],
                         "hotspot": tuple(round(x, 1) for x in f.result.metrics["hotspot_location"])} for k, f in fatigue.items()}).T
life

In [ ]:
worst = life["damage_per_mission"].astype(float).idxmax()
tviz.show(tviz.plot_damage(fatigue[worst], unit_results["lift"].artifacts["mesh"]))
contrib = pd.Series(fatigue[worst].contributions).sort_values(ascending=False)
ax = contrib.head(8).plot.barh(figsize=(8, 3.2), title=f"{worst}: damage contributions at the hotspot"); ax.invert_yaxis()

In [ ]:
hot = fatigue[worst].hotspot
unit_vm = {k: float(talos.read_frd(r.artifacts["frd"]).von_mises[hot]) / UNIT[k][1] for k, r in unit_results.items()}
rows = {}
for k, sp in spectra.items():
    peak = {}
    for b in sp.blocks:
        peak[b.pattern] = max(peak.get(b.pattern, 0.0), abs(b.mean) + abs(b.amplitude))
    stress = sum(peak.get(pat, 0.0) * unit_vm[pat] for pat in unit_vm)
    rows[k] = {**{f"peak_{pat}": peak.get(pat, 0.0) for pat in unit_vm}, "hotspot_stress_MPa": stress, "SF_yield": LW_PLA.yield_strength / stress}
pd.DataFrame(rows).T.round(2)

## 6. What the numbers say — and the vibration amplitude the resonance really causes

Read the damage column first: at ~10⁻¹⁵ per mission this wing is **not fatigue-limited** — the
solid-equivalent stresses are a fraction of a MPa against an S-N knee of tens of MPa. That is a
result, not a failure of the method: the spectrum, the hotspot and the ranking of the missions are
recorded, and the same pipeline will bite when the wing becomes a 1.2 mm printed shell (the stresses
scale with the section modulus ratio, an input for the next revision).

What the resonance *does* cause is motion: the nacelle oscillates at the shaft frequency with an
amplitude = amplification × unbalance force × static compliance. That is the number to compare with
what the camera and the propeller bearings tolerate. The table gives it per operating point for the
wing as designed, with the resonance avoided (cruise rpm moved), and for the thinner NACA 2412 wing.

In [ ]:
damage = {k: f.result.metrics["damage_per_pass"] for k, f in fatigue.items()}
hours = {k: m.duration_h for k, m in missions.items()}
usage = {"survey": 0.5, "patrol": 0.3, "windy_hops": 0.2}
rate = sum(usage[k] * damage[k] for k in usage) / sum(usage[k] * hours[k] for k in usage)
print(f"damage per 1000 flight hours with usage {usage}: {rate * 1000:.3g}  ->  "
      f"{'not life-limiting (> 1e6 h)' if rate * 1e6 < 1 else f'{1 / rate:.0f} h to failure'}")
sim = chronos.simulate_life(damage, hours, usage, n_flights=2000, seed=0)
fig = sim.plot()

In [ ]:
def assess_design(step, label):
    # a new mesh, modal analysis and unit cases for another wing STEP; same regions, masses, missions, curve
    work = RUNS / label
    m0 = model([], "modal", step); m0.mesh(work / "mesh")
    md_ = m0.solve_modes(work / "mesh", n_modes=8)
    st = chronos.Structure(tuple(md_.metrics["frequencies_hz"]), 0.03)
    units = {}
    for name, (mm, load) in UNIT.items():
        shutil.copytree(work / "mesh", work / name)
        units[name] = (model(mm.loads, name, step).solve(work / name), load)
    dmg = {kk: talos.assess_fatigue(units, chronos.build_spectrum(mission, st).to_dict(), CURVE).result.metrics["damage_per_pass"]
           for kk, mission in missions.items()}
    return st, units, dmg

no_resonance = {kk: talos.assess_fatigue(unit_cases, chronos.build_spectrum(m, None).to_dict(), CURVE).result.metrics["damage_per_pass"]
                for kk, m in missions.items()}
thin_step = drone.generate(part="wing", thickness=0.12).export_step(RUNS / "cad" / "wing_2412.step")
thin_structure, thin_units, thin_damage = assess_design(thin_step, "naca2412")
print("NACA 2412 modes:", [round(x, 1) for x in thin_structure.modes_hz])

def nacelle_amplitude_mm(st, units, point):
    compliance = units["vib_left"][0].metrics["max_displacement"] / UNIT["vib_left"][1]        # mm per N, static
    f = PROP[point]["rpm"] / 60
    daf = float(st.amplification(f)[0]) if st is not None else 1.0
    return daf * PROP[point]["unbalance_N"] * compliance

variants = {"NACA 2415 (as is)": (structure, unit_cases, damage), "NACA 2415, cruise rpm moved off the mode": (None, unit_cases, no_resonance),
            "NACA 2412 (thinner)": (thin_structure, thin_units, thin_damage)}
rows = {}
for name, (st, units, dmg) in variants.items():
    r = {f"nacelle amplitude {pt} [mm]": nacelle_amplitude_mm(st, units, pt) for pt in PROP}
    r["damage per 1000 h (usage mix)"] = sum(usage[kk] * dmg[kk] for kk in usage) / sum(usage[kk] * hours[kk] for kk in usage) * 1000
    rows[name] = r
pd.DataFrame(rows).T

## 7. Export

In [ ]:
summary = {"design": {"file": "designs/fixed_wing.py", "parameters": p}, "modes_hz": freqs, "damping_ratio": 0.03,
           "excitations_hz": lines, "margins": margins.to_dict(),
           "unit_cases": {k: {"load": UNIT[k][1], "max_von_mises": unit_results[k].metrics["max_von_mises"]} for k in UNIT},
           "curve": CURVE.__dict__, "missions": {k: m.describe() for k, m in missions.items()},
           "damage_per_mission": damage, "damage_no_resonance": no_resonance, "damage_naca2412": thin_damage,
           "hours_per_mission": hours, "usage": usage, "damage_per_1000h": rate * 1000, "nacelle_amplitude_mm": rows,
           "spectra": {k: str(RUNS / f"spectrum_{k}.json") for k in missions}}
(RUNS / "life.json").write_text(json.dumps(summary, indent=2, default=float))
print("written:", sorted(x.name for x in RUNS.iterdir()))

**Reading the result:** the amplitude column is the design lever — a mode on the cruise shaft
frequency turns a 0.2 N unbalance into millimetres of nacelle motion, and moving the rpm (or the mode)
removes it; the damage column says fatigue is not what limits this wing. Both are recorded with the
missions and curves that produced them. The decision, and the revision that follows, is yours — this
notebook only makes it visible and repeatable.